# 📊 Dərs 2: LangChain və Ragas ilə RAG Sisteminin Qiymətləndirilməsi (RAG Metrics)

Bu praktiki dərsdə **Leytenant Komandir Data (Lt. Cmdr. Data)** RAG çatbotunu yenidən quracağıq, lakin bu dəfə iki mühüm yeniliklə:

1. **LangChain Kitabxanası və FAISS Vektor Bazası:** Əvvəlki dərsdə bütün addımları sıfırdan yazmışdıqsa, burada sənayedə geniş istifadə olunan **LangChain (LCEL — LangChain Expression Language)** və **FAISS** vektor axtarış mühərrikindən, həmçinin rəsmi **Google GenAI (`from google.genai import types`)** SDK-sından istifadə edəcəyik.
2. **Ragas ilə Obyektiv Qiymətləndirmə (RAG Triad & Metrics):** Qurduğumuz RAG sisteminin nə dərəcədə keyfiyyətli işlədiyini **Ragas** kitabxanası vasitəsilə 5 əsas metrika üzrə rəqəmsal olaraq ölçəcəyik:
   - **Faithfulness (Sədaqətlilik / Dəqiqlik):** Model cavab verərkən yalnız tapılmış kontekstə əsaslanırmı, yoxsa özündən fakt uydururmu (*hallüsinasiya*)?
   - **Answer Relevancy (Cavabın Uyğunluğu):** Verilən cavab istifadəçinin sualına nə dərəcədə hədəflidir?
   - **Context Precision (Kontekst Dəqiqliyi):** Tapılmış 10 kontekst sətrindən həqiqətən doğru faktları ehtiva edənlər siyahının yuxarısında (ilk sıralarda) yerləşirmi?
   - **Context Recall (Kontekst Tamlığı):** Etalon cavab (*Ground Truth*) üçün lazım olan bütün məlumatlar bazadan tapılıbmı?
   - **Answer Correctness (Cavabın Doğruluğu):** Modelin verdiyi yekun cavab etalon (*ground truth*) cavabla həm faktiki, həm də semantik cəhətdən nə qədər üst-üstə düşür?

Bu göstəricilər bizə baza (*benchmark*) nəticəsi verəcək ki, növbəti dərslərdə (*Advanced RAG*) etdiyimiz təkmilləşdirmələrin faydasını dəqiq müqayisə edə bilək.

İlk olaraq əvvəlki dərsdə olduğu kimi `./sample_data/tng` qovluğundakı orijinal ssenariləri oxuyaraq **DATA** obrazının replikalarını çıxaraq:

In [ ]:
import os
import re
import random
import sys

# Lokal mühitdə köhnə 'google-colab' paketinin digər kitabxanalarda xəta verməsinin qarşısını alırıq:
sys.modules["google.colab"] = None

# Bütün çıxarılmış DATA replikalarının saxlanacağı əsas siyahı
dialogues = []

def strip_parentheses(s):
    """Mötərizə daxilindəki səhnə qeydlərini mətndən silir: məs. '(hesitates)'"""
    return re.sub(r'\(.*?\)', '', s)

def is_single_word_all_caps(s):
    """Sətrin tək sözdən ibarət və BÖYÜK HƏRFLƏRLƏ yazılmış obraz adı olub-olmadığını yoxlayır"""
    words = s.split()
    if len(words) != 1:
        return False
    if bool(re.search(r'\d', words[0])):
        return False
    return words[0].isupper()

def extract_character_lines(file_path, character_name):
    """Verilmiş ssenari faylından konkret obrazın (DATA) bütün replikalarını çıxarır"""
    lines = []
    with open(file_path, 'r') as script_file:
        try:
            lines = script_file.readlines()
        except UnicodeDecodeError:
            pass

    is_character_line = False
    current_line = ''
    current_character = ''
    for line in lines:
        strippedLine = line.strip()
        if is_single_word_all_caps(strippedLine):
            is_character_line = True
            current_character = strippedLine
        elif (line.strip() == '') and is_character_line:
            is_character_line = False
            dialog_line = strip_parentheses(current_line).strip()
            dialog_line = dialog_line.replace('"', "'")
            if current_character == character_name and len(dialog_line) > 0:
                dialogues.append(dialog_line)
            current_line = ''
        elif is_character_line:
            current_line += line.strip() + ' '

def process_directory(directory_path, character_name):
    """Qovluqdakı bütün ssenari fayllarını ardıcıl emal edir"""
    for filename in os.listdir(directory_path):
        file_path = os.path.join(directory_path, filename)
        if os.path.isfile(file_path):
            extract_character_lines(file_path, character_name)


In [ ]:
# './sample_data/tng' qovluğundakı bütün ssenarilərdən 'DATA' obrazının replikalarını çıxarırıq
process_directory("./sample_data/tng", 'DATA')

### ✅ Məlumatların Yoxlanılması (Sanity Check)

Replikaların düzgün oxunduğundan əmin olmaq üçün ilk sətri (`dialogues[0]`) və ümumi sayını ekrana çap edək:

In [ ]:
print(f"Ümumi replika sayı: {len(dialogues)}")
print("İlk replika:", dialogues[0])

### 📦 Lazımi Kitabxanaların Quraşdırılması (`google-genai`, `ragas`, `langchain`, `faiss-cpu`)

RAG modelimiz üçün rəsmi **Google GenAI SDK (`google-genai`)**, zəncirvari boru kəməri üçün **LangChain**, vektor axtarışı üçün **FAISS** və nəticələri ölçmək üçün **Ragas** kitabxanalarını quraşdıraq:

In [ ]:
# Google GenAI SDK-nı yeniləyirik
!pip install google-genai python-dotenv --upgrade

İndi isə **Ragas**, **LangChain** və **Hugging Face `datasets`** paketlərini quraşdıraq:

In [ ]:
# LangChain + Ragas + Datasets kitabxanalarını quraşdırırıq
!pip install -U ragas langchain langchain-core langchain-community datasets nest_asyncio


### 🔑 Addım 1: Google Gemini API (`from google.genai import types`) və LangChain Adapterlərinin Qurulması

Burada rəsmi **`google-genai`** kitabxanasından (`from google import genai` və `from google.genai import types`) istifadə edərək:
1. API açarını mühit dəyişənindən (`GEMINI_API_KEY` / `GOOGLE_API_KEY`), `.env` faylından və ya birbaşa daxiletmədən (`getpass`) oxuyuruq.
2. **`GeminiEmbeddings`** və **`ChatGemini`** siniflərini yaradırıq ki, həm **LangChain (`FAISS`, `rag_chain`)**, həm də **Ragas (`evaluate`)** birbaşa `client.models.embed_content` (`types.EmbedContentConfig`) və `client.models.generate_content` (`types.GenerateContentConfig`) metodlarımızla problemsiz işləsin (daxilində avtomatik *Rate Limit / 429* gözləmə mexanizmi ilə birlikdə).

In [ ]:
import os
import time
import asyncio
from typing import List, Optional, Any
from getpass import getpass

from google import genai
from google.genai import types
from langchain_core.embeddings import Embeddings
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import BaseMessage, AIMessage, SystemMessage
from langchain_core.outputs import ChatResult, ChatGeneration

# Əgər layihədə .env faylı varsa, açarları avtomatik yükləyirik
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# 1. Gemini API açarını mühitdən oxuyuruq və ya istifadəçidən istəyirik
api_key = os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY')
if not api_key:
    api_key = getpass('Gemini API açarınızı daxil edin (AIzaSy...): ').strip().strip('"').strip("'")

# 2. Rəsmi Google GenAI Client yaradırıq
client = genai.Client(api_key=api_key)

# 3. Google GenAI SDK (`types.EmbedContentConfig`) əsasında LangChain Embeddings sinfi
class GeminiEmbeddings(Embeddings):
    """google-genai və types.EmbedContentConfig istifadə edən LangChain Embeddings adapteri"""
    def __init__(self, model: str = "gemini-embedding-001", dimensions: int = 128):
        self.model = model
        self.dimensions = dimensions

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        all_embeddings = []
        batch_size = 100  # Gemini API bir sorğuda maksimum 100 mətn qəbul edir
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            while True:
                try:
                    response = client.models.embed_content(
                        model=self.model,
                        contents=batch,
                        config=types.EmbedContentConfig(
                            output_dimensionality=self.dimensions
                        )
                    )
                    all_embeddings.extend([e.values for e in response.embeddings])
                    break
                except Exception as e:
                    if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
                        print("Embedding API sorğu limiti (429), 20 saniyə gözlənilir...")
                        time.sleep(20)
                    else:
                        raise e
        return all_embeddings

    def embed_query(self, text: str) -> List[float]:
        return self.embed_documents([text])[0]

    async def aembed_documents(self, texts: List[str]) -> List[List[float]]:
        return await asyncio.to_thread(self.embed_documents, texts)

    async def aembed_query(self, text: str) -> List[float]:
        return await asyncio.to_thread(self.embed_query, text)


# 4. Google GenAI SDK (`types.GenerateContentConfig`) əsasında LangChain ChatModel sinfi
class ChatGemini(BaseChatModel):
    """google-genai və types.GenerateContentConfig istifadə edən LangChain Chat adapteri"""
    model_name: str = "gemini-3.8-flash"
    temperature: float = 0.0

    @property
    def _llm_type(self) -> str:
        return "google_genai_chat"

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[Any] = None,
        **kwargs: Any,
    ) -> ChatResult:
        sys_parts = []
        user_parts = []
        for m in messages:
            if isinstance(m, SystemMessage) or getattr(m, 'type', '') == 'system':
                sys_parts.append(str(m.content))
            else:
                user_parts.append(str(m.content))

        sys_instruction = "\n".join(sys_parts) if sys_parts else None
        prompt_text = "\n\n".join(user_parts) if user_parts else " "

        while True:
            try:
                response = client.models.generate_content(
                    model=self.model_name,
                    contents=prompt_text,
                    config=types.GenerateContentConfig(
                        system_instruction=sys_instruction,
                        temperature=self.temperature
                    )
                )
                text = response.text if response.text else ""
                return ChatResult(generations=[ChatGeneration(message=AIMessage(content=text))])
            except Exception as e:
                if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
                    print("Chat API sorğu limiti (429), 15 saniyə gözlənilir...")
                    time.sleep(15)
                else:
                    raise e

    async def _agenerate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[Any] = None,
        **kwargs: Any,
    ) -> ChatResult:
        return await asyncio.to_thread(self._generate, messages, stop=stop, run_manager=run_manager, **kwargs)

print("Google Gemini Client və LangChain adapterləri uğurla hazırlandı!")

### 📝 Addım 2: Replikaların Mətn Faylına Yazılması (`data_lines.txt`)

`LangChain` standart olaraq sənədləri müəyyən uzunluqlu parçalara (*chunks*) bölməyə çalışır. Biz isə əvvəlki dərsdəki üsulla tam eyni müqayisə aparmaq üçün **hər bir replika sətrini ayrıca bir sənəd (1 sətir = 1 sənəd)** kimi saxlamaq istəyirik.

> 📌 **Vacib Qeyd (Gemini Pulsuz Kvota / Free Tier Limiti):**
> Google Gemini-nin pulsuz planında `embed_content` dəqiqədə maksimum **100 mətn** qəbul etdiyi üçün, burada **"Lal" haqqındakı bütün replikaları (47 sətir) + digər ümumi replikaları əhatə edən 100 sətri (`MAX_LINES = 100`)** `./sample_data/data_lines.txt` faylına yazırıq ki, `FAISS` vektor bazası 1 saniyə ərzində limitə düşmədən yaradılsın *(ödənişli planda `MAX_LINES = len(dialogues)` edə bilərsiniz)*.

In [ ]:
# Pulsuz API kvotasında (dəqiqədə 100 mətn) dərhal işləməsi üçün MAX_LINES = 100 seçirik.
# (Ödənişli planda bütün sətirlər üçün MAX_LINES = len(dialogues) yaza bilərsiniz)
MAX_LINES = 100

if MAX_LINES < len(dialogues):
    lal_lines = [line for line in dialogues if 'Lal' in line]
    other_lines = [line for line in dialogues if 'Lal' not in line]
    selected_dialogues = (lal_lines + other_lines)[:MAX_LINES]
else:
    selected_dialogues = dialogues

# Seçilmiş replikaları LangChain-in oxuya bilməsi üçün tək bir mətn faylına yazırıq:
with open("./sample_data/data_lines.txt", "w+", encoding="utf-8") as f:
    for line in selected_dialogues:
        f.write(line + "\n")

print(f"'./sample_data/data_lines.txt' faylına {len(selected_dialogues)} replika yazıldı.")

### 📄 Addım 3: Fərdi Sənəd Yükləyicisinin (`CustomDocumentLoader`) Yazılması

İndi `data_lines.txt` faylını sətir-sətir oxuyub hər bir sətri ayrıca `Document` obyektinə çevirən `CustomDocumentLoader` sinfini yazırıq.

`LangChain`-də bunu hazır edən sadə funksiya yoxdur — əslində bu, `LangChain`-in bizə *"mətni tək-tək sətirlər şəklində saxlamaq bəlkə də ən yaxşı ideya deyil"* mesajıdır (bunun səbəbini az sonra Ragas metrikalarında aydın görəcəyik!).

In [ ]:
# LangChain sənədlərindən nümunə əsasında fərdi yükləyici (CustomDocumentLoader)
from typing import Iterator
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document


class CustomDocumentLoader(BaseLoader):
    """Faylı sətir-sətir oxuyub hər sətri müstəqil LangChain Document obyektinə çevirən yükləyici."""

    def __init__(self, file_path: str) -> None:
        """Yükləyicini fayl yolu (file_path) ilə inisializasiya edir."""
        self.file_path = file_path

    def lazy_load(self) -> Iterator[Document]:
        """Yaddaşa qənaət etmək üçün generator (yield) vasitəsilə sətirləri tək-tək Document kimi qaytarır."""
        with open(self.file_path, encoding="utf-8") as f:
            line_number = 0
            for line in f:
                yield Document(
                    page_content=line,
                    metadata={"line_number": line_number, "source": self.file_path},
                )
                line_number += 1

### 🗄️ Addım 4: FAISS Vektor Verilənlər Bazasının Yaradılması

İndi işimiz xeyli sadələşir! İlk öncə Meta tərəfindən yaradılmış sürətli vektor axtarış kitabxanası olan **`faiss-cpu`** paketini quraşdırırıq, ardınca cəmi 3 sətir kodla sənədləri yükləyib `GeminiEmbeddings` vasitəsilə **FAISS** vektor bazasını qururuq:

In [ ]:
# Sürətli lokal vektor axtarışı üçün FAISS kitabxanasını quraşdırırıq
!pip install --upgrade faiss-cpu

In [ ]:
from langchain_community.vectorstores import FAISS

# 1. Fərdi yükləyici (CustomDocumentLoader) ilə dialoq sətirlərini LangChain Document kimi yükləyirik
loader = CustomDocumentLoader("./sample_data/data_lines.txt")
docs = list(loader.lazy_load())

# 2. Google Gemini Embeddings (gemini-embedding-001) yaradırıq və FAISS vektor bazasını qururuq
embeddings = GeminiEmbeddings(model="gemini-embedding-001", dimensions=128)
vectorstore = FAISS.from_documents(docs, embeddings)

print(f"Loaded {len(docs)} documents into FAISS vector store.")


### 🔗 Addım 5: LangChain RAG Boru Kəmərinin (`rag_chain`) Qurulması

İndi RAG zəncirini (*pipeline*) quraq. Əvvəlki dərsdən fərqli olaraq:
- Modelə **Leytenant Komandir Data** kimi davranmaq təlimatını birbaşa **Sistem Təlimatında (`system_prompt`)** veririk və cavabı maksimum 3 cümlə ilə yığcam saxlamasını, bilmədiyi suallara isə *"bilmirəm"* deməsini tapşırırıq.
- Əvvəlki nümunə ilə eyni olması üçün `search_kwargs={"k": 10}` təyin edərək vektor bazasından **Top-10** kontekst sətrini gətiririk.
- `|` operatoru (*LCEL — LangChain Expression Language*) ilə axtarış (`retriever`), şablon (`prompt`), model (`llm`) və mətn çıxarıcını (`StrOutputParser`) zəncirvari birləşdiririk.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Google Gemini Chat modeli (temperature=0 -> deterministik və dəqiq cavablar üçün)
llm = ChatGemini(model_name="gemini-3.8-flash", temperature=0.0)

# 2. Sistem təlimatı (System Prompt)
system_prompt = (
    "You are Lt. Commander Data from Star Trek: The Next Generation. "
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise."
)

# 3. Sual və konteksti birləşdirən ChatPromptTemplate şablonu
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}\n\nContext:\n{context}")
])

# 4. Vektor bazasından ən yaxın 10 sənədi (k=10) gətirən Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# 5. LCEL (|) sintaksisi ilə tam RAG zəncirinin qurulması
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


### 🧪 Addım 6: RAG Zəncirinin Sınaqdan Keçirilməsi

Əvvəlki dərsdəki eyni sualla (`"Tell me about your daughter, Lal."`) həm tapılan 10 mənbə sənədini (`SOURCE DOCUMENTS`), həm də modelin verdiyi yekun cavabı (`RESULT`) yoxlayaq:

In [ ]:
question = "Tell me about your daughter, Lal."

# 1. Retriever vasitəsilə suala ən uyğun 10 kontekst sənədini gətiririk
docs = retriever.invoke(question)

print("SOURCE DOCUMENTS:\n")
for i, doc in enumerate(docs, start=1):
    print(f"[DOC {i}]")
    print(doc.page_content)
    print("-" * 80)

# 2. RAG zəncirini işə salaraq yekun cavabı alırıq
print("\nRESULT:\n")
answer = rag_chain.invoke(question)
print(answer)


### 📏 Addım 7: Ragas Qiymətləndirməsi Üçün Test Sualları və Etalon Cavablar (`Ground Truth`)

İndi RAG modelimizin nə qədər yaxşı işlədiyini **Ragas** ilə rəqəmsal olaraq ölçək!

Bunun üçün:
- **`eval_questions`:** 6 ədəd sınaq sualı tərtib edirik.
- **`eval_answers` (Ground Truth):** `context_recall` və `answer_correctness` kimi metrikalar modelin cavabını həqiqi doğru cavabla müqayisə etdiyi üçün hər sualın qarşısında ideal **etalon cavabları** müəyyən edirik.

In [ ]:
# Sınaq sualları (Test Questions)
eval_questions = [
    "Is Lal your daughter?",
    "How many calculations per second can Lal complete?",
    "Does Lal have emotions?",
    "What goal did you have for Lal?",
    "How was Lal's species and gender chosen?",
    "What happened to Lal?"
]

# Hər bir sual üçün həqiqi etalon cavablar (Ground Truth Answers)
eval_answers = [
    "Yes, Lal is my daughter. I created Lal.",
    "Lal is capable of completing sixty trillion calculations per second.",
    "Yes, unlike myself, Lal proved able to feel emotions such as fear and love.",
    "My goal for Lal was for her to enter Starfleet Academy.",
    "Lal chose her own identity as a human female, after consulting with Counselor Troi.",
    "Lal experienced a cascade failure in her neural net, triggered by distress from her impending separation from me to Galor IV. I deactivated Lal once she suffered complete neural system failure."
]


### 🔍 Tək Bir Sual Üzərində Yoxlama

Bütün siyahını işə salmazdan əvvəl 2-ci sual (`"How many calculations per second can Lal complete?"`) üzərində RAG zəncirimizin necə cavab verdiyinə baxaq:

In [ ]:
# 2-ci sual üzərində sürətli yoxlama:
test_q = eval_questions[1]

print("QUESTION:")
print(test_q)
print("\nANSWER:")
print(rag_chain.invoke(test_q))


### 📥 Addım 8: Bütün Test Sualları Üzrə Cavabların (`answers`) və Kontekstlərin (`contexts`) Toplanması

Ragas qiymətləndirməsi üçün test sualları və etalon cavablarla yanaşı:
1. Modelin hər suala verdiyi **cavabı (`answers`)**
2. Həmin cavabı yaratmaq üçün vektor bazasından gətirilmiş **kontekst mətnlərini (`contexts`)**

toplayırıq:

In [ ]:
answers = []
contexts = []

for q in eval_questions:
    # 1) Sual üçün vektor bazasından tapılan kontekst sənədləri
    ctx_docs = retriever.invoke(q)

    # 2) RAG zəncirinin generasiya etdiyi cavab
    ans = rag_chain.invoke(q)

    answers.append(ans)
    # Ragas 'contexts' üçün sətirlər siyahısı (list[str]) tələb edir
    contexts.append([doc.page_content for doc in ctx_docs])

len(answers), len(contexts)


### 🗂️ Addım 9: Məlumatların Hugging Face `Dataset` Formatına Çevrilməsi

Ragas kitabxanası qiymətləndirmə aparmaq üçün məlumatları **Hugging Face `Dataset`** formatında qəbul edir (`question`, `answer`, `contexts`, `ground_truth` sütunları ilə). Gəlin bu cədvəli yaradaq və ilk sətrin strukturuna baxaq:

In [ ]:
from datasets import Dataset

response_dataset = Dataset.from_dict({
    "question": eval_questions,
    "answer": answers,          # RAG modelimizin yaratdığı cavablar
    "contexts": contexts,       # Vektor bazasından tapılmış kontekst siyahıları (list[list[str]])
    "ground_truth": eval_answers # Bizim təyin etdiyimiz ideal etalon cavablar
})

# İlk sualın dataset strukturuna baxırıq
response_dataset[0]


### ⚖️ Addım 10: Ragas Metrikalarının və Hakim Modelin (`Evaluator LLM`) Təyin Edilməsi

İndi Ragas-a hansı 5 metrikanı ölçmək istədiyimizi bildiririk və yuxarıda yaratdığımız Google Gemini (`llm` və `embeddings`) obyektlərini Ragas-ın hakim modeli (*LLM-as-a-Judge*) kimi bükürük:

In [ ]:
from ragas import evaluate, RunConfig
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    answer_correctness,
    context_recall,
    context_precision,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Ölçmək istədiyimiz 5 əsas RAG metrikası
metrics = [
    faithfulness,       # Cavab kontekstə nə qədər sadiqdir (hallüsinasiya yoxlaması)
    answer_relevancy,   # Cavab suala nə qədər uyğundur
    context_recall,     # Etalon cavab üçün lazım olan faktlar kontekstdə tapılıbmı
    context_precision,  # Doğru kontekstlər siyahıda yuxarı sıralardadırmı
    answer_correctness, # Yekun cavab etalon cavabla nə qədər üst-üstə düşür
]

# Google Gemini LLM və Embeddings obyektlərimizi Ragas üçün bükürük
evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)


### 🚀 Addım 11: `evaluate()` Funksiyasının İşə Salınması

İndi `evaluate()` funksiyasını çağıraraq qiymətləndirməni başladırıq!

> 💡 **Qeyd:** `RunConfig(max_workers=2)` parametri Ragas-ın eyni anda həddən artıq paralel sorğu göndərərək API limitini aşmasının qarşısını alır.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

results = evaluate(
    response_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=RunConfig(max_workers=2, timeout=180)
)

### 📈 Addım 12: Qiymətləndirmə Nəticələrinin Təhlili

Gəlin həm ümumi orta xallara (`results`), həm də hər bir sual üzrə detallı Pandas cədvəlinə (`results.to_pandas()`) baxaq!

Görəcəyimiz kimi, ssenarini **tək-tək sətirlər** şəklində saxladıqda bəzi suallarda (məsələn, *"Lal saniyədə neçə hesablama aparır?"*) cümlədə *"Lal"* adı əvəzinə *"o" (she)* əvəzliyi işləndiyi üçün axtarış həmin sətri tapa bilmir və `context_recall` aşağı düşür. Növbəti dərsdə (*Advanced RAG*) bunu necə həll edəcəyimizi öyrənəcəyik!

In [ ]:
# Bütün suallar üzrə orta (benchmark) metrika nəticələri
results

In [ ]:
# Hər bir sual üzrə detallı qiymətləndirmə cədvəli
results.to_pandas()